In [1]:
import pandas as pd  
import numpy as np  
import os  
import re  
import glob  
import io
import calendar
from datetime import datetime  
import msoffcrypto
import warnings
from dateutil.relativedelta import relativedelta
from collections import defaultdict
from dotenv import load_dotenv  


warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [2]:
import os, io, glob  
from collections import defaultdict  
import msoffcrypto  
import pandas as pd  
from dotenv import load_dotenv

load_dotenv()  
password = os.environ['EPIC_XL_PW']

BASE_TEMPLATE = r'L:/2026 Pilar Plant Files/AP EPIC Audit/{site}/Backup/EPIC - Received from Meaghan'  
SITES = ['LENOX', 'GVCCC', 'MEETH']


def read_protected_excel(file_path, password):  
    """Return {sheet_name: DataFrame} from a password-protected workbook."""  
    decrypted = io.BytesIO()  
    with open(file_path, 'rb') as f:  
        office_file = msoffcrypto.OfficeFile(f)  
        if office_file.is_encrypted():  
            office_file.load_key(password=password)  
            office_file.decrypt(decrypted)  
        else:  
            decrypted.write(f.read())   # handle any unprotected files  
    decrypted.seek(0)  
    return pd.read_excel(decrypted, sheet_name=None)


def load_site(site, password):  
    """Load every tab of every workbook for one site.  
    Returns (by_sheet_dict, stacked_df)."""  
    base_path = BASE_TEMPLATE.format(site=site)  
    print(f"\n=== {site} ===")

    if not os.path.isdir(base_path):  
        print(f"  !! path not found: {base_path}")  
        return {}, pd.DataFrame()

    epic_files = [  
        p for p in glob.glob(os.path.join(base_path, '*.xls*'))  
        if not os.path.basename(p).startswith('~$')  
    ]  
    print(f"  {len(epic_files)} file(s) found")

    by_sheet = defaultdict(list)  
    for file_path in epic_files:  
        filename = os.path.basename(file_path)  
        try:  
            sheets = read_protected_excel(file_path, password)  
            for sheet_name, df in sheets.items():  
                df.insert(0, 'site', site)  
                df.insert(1, 'source_file', filename)  
                df.insert(2, 'source_sheet', sheet_name)  
                by_sheet[sheet_name].append(df)  
            print(f"  OK: {filename} ({len(sheets)} tabs)")  
        except Exception as e:  
            print(f"  ERROR in {filename}: {e}")

    by_sheet = {name: pd.concat(frames, ignore_index=True)  
                for name, frames in by_sheet.items()}

    stacked = (pd.concat(by_sheet.values(), ignore_index=True)  
               if by_sheet else pd.DataFrame())

    print(f"  Tabs found: {list(by_sheet)}")  
    for name, df in by_sheet.items():  
        print(f"    {name}: {df.shape[0]:,} rows x {df.shape[1]} cols")

    return by_sheet, stacked


# ---- run all three sites and stack them ----  
results = {site: load_site(site, password) for site in SITES}

epic_all = pd.concat(  
    [stacked for _, stacked in results.values() if not stacked.empty],  
    ignore_index=True  
)

print("\n=== SUMMARY ===")  
print(epic_all.groupby(['site', 'source_sheet']).size().to_string())  
print(f"\nCOMBINED: {epic_all.shape[0]:,} rows x {epic_all.shape[1]} cols")  


=== LENOX ===
  4 file(s) found
  OK: LHH August 2026 Implants.xlsx (1 tabs)
  OK: LHH July 2026 Implants.xlsx (1 tabs)
  OK: LHH Nov - Jun 2026 FULL.xlsx (1 tabs)
  OK: LHH September 2026 Implants.xlsx (1 tabs)
  Tabs found: ['Sheet1']
    Sheet1: 131,478 rows x 25 cols

=== GVCCC ===
  6 file(s) found
  OK: GV OR Endo and Cath Lab Implant Feb - Apr 2026.xlsx (1 tabs)
  OK: GV OR Endo and Cath Lab Implant May - June 2026.xlsx (1 tabs)
  OK: GV OR Endo and Cath Lab Implant Nov - Jan 2026.xlsx (1 tabs)
  OK: GVH Implants August 2026.xlsx (1 tabs)
  OK: GVH Jul 2026 Implants.xlsx (1 tabs)
  OK: GVH Spet 2026 Implants.xlsx (1 tabs)
  Tabs found: ['Sheet1']
    Sheet1: 19,375 rows x 25 cols

=== MEETH ===
  6 file(s) found
  OK: MEETH August Implants.xlsx (1 tabs)
  OK: MEETH Implants Mar - May.xlsx (1 tabs)
  OK: MEETH Implants Nov - Feb.xlsx (1 tabs)
  OK: MEETH July Implants.xlsx (1 tabs)
  OK: MEETH June Implants.xlsx (1 tabs)
  OK: MEETH September Implants.xlsx (1 tabs)
  Tabs found:

In [11]:
import os, glob  
import pandas as pd

AP_TEMPLATE = r'L:/2026 Pilar Plant Files/AP EPIC Audit/{site}/Backup/AP'  
SITES = ['LENOX', 'GVCCC', 'MEETH']


def load_site_ap(site, sheet_name='Voucher Drill', skiprows=3):  
    """Read the Voucher Drill tab from every AP workbook for one site."""  
    ap_path = AP_TEMPLATE.format(site=site)  
    print(f"\n=== {site} ===")

    if not os.path.isdir(ap_path):  
        print(f"  !! path not found: {ap_path}")  
        return pd.DataFrame()

    ap_files = [  
        p for p in glob.glob(os.path.join(ap_path, '*.xlsx'))  
        if not os.path.basename(p).startswith('~$')  
    ]  
    print(f"  Found {len(ap_files)} files")

    ap_data = []  
    for file in ap_files:  
        filename = os.path.basename(file)  
        try:  
            file_data = pd.read_excel(file, sheet_name=sheet_name, skiprows=skiprows)  
            file_data.insert(0, 'site', site)            # site first  
            file_data.insert(1, 'source_file', filename) # then file  
            ap_data.append(file_data)  
            print(f"  Processed: {filename} ({len(file_data):,} rows)")

        except PermissionError:  
            print(f"  SKIPPED (permission denied): {filename}")  
        except ValueError as e:  
            print(f"  SKIPPED (no '{sheet_name}' tab): {filename} -- {e}")  
        except Exception as e:  
            print(f"  ERROR in {filename}: {e}")

    if not ap_data:  
        print("  No files loaded")  
        return pd.DataFrame()

    combined = pd.concat(ap_data, ignore_index=True)  
    print(f"  Total rows: {len(combined):,} x {combined.shape[1]} cols")  
    return combined


# ---- run all three sites and stack them ----  
frames = [load_site_ap(site) for site in SITES]  
ap_all = pd.concat([df for df in frames if not df.empty], ignore_index=True)

print("\n=== SUMMARY ===")  
print(ap_all.groupby('site').size().to_string())  
print(f"\nCOMBINED: {len(ap_all):,} rows x {ap_all.shape[1]} cols")  


=== LENOX ===
  Found 3 files
  Processed: Lenox voucher detail 11 2025.xlsx (7,238 rows)
  Processed: Lenox voucher detail 12 2025.xlsx (7,932 rows)
  Processed: Lenox voucher detail YTD 9 2026.xlsx (69,934 rows)
  Total rows: 85,104 x 53 cols

=== GVCCC ===
  Found 0 files
  No files loaded

=== MEETH ===
  Found 0 files
  No files loaded

=== SUMMARY ===
site
LENOX    85104

COMBINED: 85,104 rows x 53 cols


In [4]:
acct_xwalk = pd.read_excel('L:/2026 Pilar Plant Files/Crosswalks/Manhattan Region Crosswalk.xlsx', sheet_name='all_accounts')

dept_xwalk = pd.DataFrame(  
    [  
        ('LENOX', 15652050, 'Cardiac Cath Lab',                 'Cath'),  
        ('LENOX', 15652020, 'Cardio - EPS',                     'Cath'),  
        ('GVCCC', None,     'GVH CATH & EP LABS',               'Cath'),  
        ('GVCCC', None,     'GVH Endoscopy',                    'Endo'),  
        ('GVCCC', None,     'GVH Noninvasive Cardiology',       'Cath'),  
        ('GVCCC', None,     'GVH OR',                           'OR'),  
        ('LENOX', None,     'LHH CATH & EP LABS',               'Cath'),  
        ('LENOX', None,     'LHH Endoscopy',                    'Endo'),  
        ('LENOX', None,     'LHH NONINVASIVE CARDIOLOGY',       'Cath'),  
        ('LENOX', None,     'LHH OR',                           'OR'),  
        ('LENOX', 15625000, 'Neurosurgery',                     'Cath'),  
        ('LENOX', 15611055, 'NRSG - Cardiac cath',              'Cath'),  
        ('LENOX', 15666000, 'OP Svc - Ambulatory Surgery',      'OR'),  
        ('GVCCC', 74002548, 'Op Svc Endoscopy Center - LHGV',   'Endo'),  
        ('GVCCC', 74002545, 'POS - Operating Room',             'OR'),  
        ('LENOX', 15651060, 'POS - Endoscopy',                  'Endo'),  
        ('LENOX', 15651010, 'POS - Operating Room',             'OR'),  
        ('GVCCC', 74002545, 'POS - Operating Room',             'OR'),  
        ('MEETH', 29651010, 'POS - Operating Room',             'OR'),  
        ('GVCCC', 74002562, 'Pos-Central Services',             'OR'),  
        ('MEETH', None,     'MEETH OR',                         'OR'),  
    ],  
    columns=['BU', 'Dept', 'Dept Name', 'Category']  
)

In [5]:
epic_data_raw = epic_all[['Location', 'Date', 'Name', 'Imp PS ID', 'Manufacturer', 'Cost', 'Number Used']]

In [14]:
ap_data_spec = ap_all[['Source Type', 'GL Unit', 'Year', 'Period', 'Account', 'Dept', 'Monetary Amount', 'Vendor ID', 'Vendor Name', 'Qty', 'UOM', 'Unit Price', 'Item ID', 'Description']]

ap_depts = dept_xwalk[dept_xwalk['Dept'].notna()]
ap_data_dept = pd.merge(ap_data_spec, ap_depts[['Dept', 'Dept Name', 'Category']], how='left', on='Dept')
ap_data_acct = pd.merge(ap_data_dept, acct_xwalk, how='left', on='Account')
ap_data_full = ap_data_acct.rename(columns={'Description_x': 'Item Desc', 'Description_y': 'Acct Desc'})
ap_data_cat = ap_data_full[ap_data_full['Category'].notna()]
#ap_data_cat['Date'] = pd.PeriodIndex(year=ap_data_cat['Year'], freq='M', month=ap_data_cat['Period'])  


In [17]:
display(ap_data_cat[ap_data_cat['Item ID'] == 239579])

,Source Type,GL Unit,Year,Period,Account,Dept,Monetary Amount,Vendor ID,Vendor Name,Qty,UOM,Unit Price,Item ID,Item Desc,Dept Name,Category,Acct Desc
7528,AP,LENOX,2025.0,12.0,63067.0,15651010.0,511.70,1051.0,BAXTER BIOSCIENCE HYLAND,1.0,CA,436.697,239579.0,APPL DUPLOSPRAY 40CM 5/CA,POS - Operating Room,OR,Supplies - Implants - Urologic
8220,AP,LENOX,2025.0,12.0,63080.0,15651010.0,873.39,1051.0,BAXTER BIOSCIENCE HYLAND,2.0,CA,436.697,239579.0,APPL DUPLOSPRAY 40CM 5/CA,POS - Operating Room,OR,Supplies - Surgical
43852,AP,LENOX,2026.0,4.0,63080.0,15651010.0,524.80,1051.0,BAXTER BIOSCIENCE HYLAND,1.0,CA,449.798,239579.0,APPL DUPLOSPRAY 40CM 5/CA,POS - Operating Room,OR,Supplies - Surgical
74366,AP,LENOX,2026.0,8.0,63080.0,15651010.0,899.60,1051.0,BAXTER BIOSCIENCE HYLAND,2.0,CA,449.800,239579.0,APPL DUPLOSPRAY 40CM 5/CA,POS - Operating Room,OR,Supplies - Surgical


In [16]:


accounts =      ap_data_cat[['Item ID', 'Account', 'Acct Desc', 'Category']].drop_duplicates().reset_index(drop=True)
#vendors =       ap_data_cat.sort_values(['Item ID', 'Date'], ascending=[True, False]).groupby('Item ID')[['Vendor', 'Vendor Name']].first().reset_index()  
#item_descs =    ap_data_cat.groupby



itemids = accounts.groupby(['Item ID', 'Category']).size().reset_index(name='Count').sort_values('Count', ascending=False).reset_index(drop=True)

dupes = itemids[itemids['Count'] > 1]

if not dupes.empty:  
    print(f'{len(dupes)} Item IDs map to more than one account combination:')  
    print(dupes)
else:  
    print('All Item IDs map to exactly one account combination.') 



18 Item IDs map to more than one account combination:
     Item ID Category  Count
0   174872.0     Cath      2
1   239579.0       OR      2
2   275343.0     Cath      2
3   277010.0     Endo      2
4   278344.0       OR      2
5   294556.0     Cath      2
6   298869.0       OR      2
7   301714.0     Endo      2
8   301897.0       OR      2
9   301898.0     Cath      2
10  305422.0     Endo      2
11  305442.0       OR      2
12  305444.0       OR      2
13  305448.0       OR      2
14  308441.0     Endo      2
15  309772.0     Endo      2
16  310092.0     Endo      2
17  311136.0     Cath      2
